In [ ]:
import json
import pycountry

from emu_renewal.inputs import get_oxcgrt_data
from emu_renewal.constants import DATA_PATH
from emu_renewal.outputs import add_bool_row_to_table
from emu_renewal.selection import get_mob_avail_countries, gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats

In [ ]:
pol_data = get_oxcgrt_data()
# No data for Kosovo available from sources other than OxCGRT
pol_avail = [iso3 for iso3 in set(pol_data["CountryCode"]) if iso3 != "RKS"]
either_mob_avail, summary, _, _ = get_mob_avail_countries()
mob_or_pol_avail = list(set(pol_avail + either_mob_avail))

In [ ]:
death_data, case_data = gather_who_data(mob_or_pol_avail)
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_reps, case_reps = find_nans_repeats(death_data, case_data, summary)

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_reps + case_reps + death_outliers + case_outliers)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Included")

In [ ]:
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
summary

In [ ]:
old_included = json.load(open(DATA_PATH / "config/included.json", "r"))
len(old_included)

In [ ]:
json.dump(included, open(DATA_PATH / "config/included.json", "w"))